# Highlands Ranch, CO — LVT Policy Analysis

Models a revenue-neutral split-rate LVT shift for parcels inside the
**Highlands Ranch CDP** (Douglas County, CO).

Highlands Ranch is a Census Designated Place (CDP), not an incorporated city.
The Douglas County `Parcels_A_view` FeatureServer has a `CITY_NAME` attribute
that supports a clean string filter — no spatial join needed.

## Policy decisions (carried from PR #8)

| Choice | Setting | Note |
|---|---|---|
| Geographic scope | Highlands Ranch (`CITY_NAME = 'Highlands Ranch'`) | Douglas County attribute filter |
| Reform shape | Split-rate at 2:1 | Lars's primary scenario |
| Land/improvement values | `LANDASD` / `IMPASD` from Property_Values_Data | Pivoted from L/I valuation type codes |
| Mill levy | Per-parcel `REDUCED_MILL_LEVY`, fallback 93.1 mills | Lars's 2024 Douglas County certified breakdown — county + LE + school + library + fire + HR Metro + drainage |
| Exclusions | Exempt/Government, Agricultural | Per Lars's `analysis_gdf` filter |
| Classification | ACCTTYPE + Valuation_Description | Two-tier with zoning PD breakout |


## Section 1 — Imports and setup

In [ ]:
import sys
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from shapely.geometry import Polygon

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'highlands_ranch'
STATE_FIPS = '08'
COUNTY_FIPS = '035'  # Douglas County
MODEL_TYPE = 'split_rate_2to1'
LAND_IMPROVEMENT_RATIO = 2.0

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

PARCELS_URL = ('https://services.arcgis.com/seTexOicoRXDvRsJ/arcgis/rest/'
               'services/Parcels_A_view/FeatureServer/0/query')
OPENDATA_BASE = ('https://services.arcgis.com/seTexOicoRXDvRsJ/arcgis/rest/'
                 'services/OpenData/FeatureServer')
VALUES_URL = OPENDATA_BASE + '/4/query'  # Property_Values_Data

HR_CITY_FILTER = "CITY_NAME = 'Highlands Ranch'"
HR_DEFAULT_MILL_LEVY = 93.1
PAGE_SIZE = 2000

## Section 2 — Fetch parcels + valuation segments

1. `Parcels_A_view` — geometry + per-parcel attributes (filtered to HR)
2. `Property_Values_Data` — long table with one row per `(ACCOUNT_NO, VALUATION_TYPE_CODE)`; pivot L/I to get LANDASD / IMPASD


In [ ]:
def esri_poly(g):
    rings = (g or {}).get('rings')
    if not rings:
        return None
    return Polygon(rings[0], holes=rings[1:] if len(rings) > 1 else None)


def fetch_all(url, where='1=1', out_fields='*', return_geometry=False,
              page_size=PAGE_SIZE):
    session = requests.Session()
    count = int(session.get(url, params={
        'f': 'json', 'where': where, 'returnCountOnly': 'true'
    }, timeout=60).json().get('count', 0))
    print(f'  total: {count:,}')
    feats = []
    for off in range(0, count, page_size):
        params = {
            'f': 'json', 'where': where, 'outFields': out_fields,
            'returnGeometry': str(return_geometry).lower(),
            'resultOffset': off, 'resultRecordCount': page_size,
            'orderByFields': 'OBJECTID ASC',
        }
        if return_geometry:
            params['outSR'] = 4326
            params['geometryPrecision'] = 6
        r = session.get(url, params=params, timeout=180)
        r.raise_for_status()
        page = r.json().get('features', [])
        if not page:
            break
        feats.extend(page)
    return feats


def load_hr_parcels():
    cache = DATA_DIR / 'hr_parcels.parquet'
    if cache.exists():
        return gpd.read_parquet(cache)
    print('Fetching HR parcels...')
    feats = fetch_all(PARCELS_URL, HR_CITY_FILTER, '*', return_geometry=True)
    rows = []
    for f in feats:
        attrs = f.get('attributes', {})
        g = esri_poly(f.get('geometry') or {})
        if g is not None:
            attrs['geometry'] = g
            rows.append(attrs)
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf.to_parquet(cache, index=False)
    return gdf


def load_hr_values(hr_accounts):
    """Pull entire Property_Values_Data table (~301K rows) then filter to HR
    accounts. Pagination by IN-clause hits URL-length limits with string IDs."""
    cache = DATA_DIR / 'hr_property_values.parquet'
    if cache.exists():
        return pd.read_parquet(cache)
    accts = {str(a) for a in hr_accounts if pd.notna(a)}
    print(f'Fetching full Property_Values_Data table (paginated)...')
    feats = fetch_all(
        VALUES_URL, '1=1',
        'ACCOUNT_NO,VALUATION_TYPE_CODE,VALUATION_DESCRIPTION,VALUATION_CLASS_CODE,ACTUAL_VALUE,ASSESSED_VALUE',
        return_geometry=False,
    )
    df = pd.DataFrame([f['attributes'] for f in feats])
    df['ACCOUNT_NO'] = df['ACCOUNT_NO'].astype(str)
    df = df[df['ACCOUNT_NO'].isin(accts)].reset_index(drop=True)
    df.to_parquet(cache, index=False)
    return df


t0 = time.time()
parcels = load_hr_parcels()
print(f'HR parcels: {len(parcels):,} ({time.time()-t0:.1f}s)')

# Filter to just the columns we actually need; raw Parcels_A_view has 60+
keep = [c for c in ['OBJECTID', 'ACCOUNT_NO', 'STATE_PARCEL_NO', 'CITY_NAME',
                    'ACCOUNT_TYPE_CODE', 'PARCEL_TYPE', 'REDUCED_MILL_LEVY',
                    'TOTAL_NET_ACRES', 'OWNER_NAME', 'geometry']
        if c in parcels.columns]
parcels = parcels[keep].copy()
print(f'Parcels columns kept: {list(parcels.columns)}')

t0 = time.time()
values_df = load_hr_values(parcels['ACCOUNT_NO'])
print(f'Valuation rows: {len(values_df):,} ({time.time()-t0:.1f}s)')

In [ ]:
# Pivot L/I segments into LANDASD / IMPASD
for col in ['ACTUAL_VALUE', 'ASSESSED_VALUE']:
    values_df[col] = pd.to_numeric(values_df[col], errors='coerce').fillna(0)

segs = (values_df
        .groupby(['ACCOUNT_NO', 'VALUATION_TYPE_CODE'], dropna=False)
        .agg(seg_actual=('ACTUAL_VALUE', 'sum'),
             seg_assessed=('ASSESSED_VALUE', 'sum'))
        .reset_index())
land_seg = (segs[segs['VALUATION_TYPE_CODE'] == 'L']
            .rename(columns={'seg_actual': 'LANDACT', 'seg_assessed': 'LANDASD'})
            [['ACCOUNT_NO', 'LANDACT', 'LANDASD']])
imp_seg = (segs[segs['VALUATION_TYPE_CODE'] == 'I']
           .rename(columns={'seg_actual': 'IMPACT', 'seg_assessed': 'IMPASD'})
           [['ACCOUNT_NO', 'IMPACT', 'IMPASD']])
totals = (values_df.groupby('ACCOUNT_NO', dropna=False).agg(
    TOTALACT=('ACTUAL_VALUE', 'sum'),
    TOTALASD=('ASSESSED_VALUE', 'sum'),
    Valuation_Description=('VALUATION_DESCRIPTION', 'first'),
).reset_index())

account_values = (totals
                  .merge(land_seg, on='ACCOUNT_NO', how='left')
                  .merge(imp_seg, on='ACCOUNT_NO', how='left'))
for col in ['LANDACT', 'LANDASD', 'IMPACT', 'IMPASD']:
    account_values[col] = pd.to_numeric(account_values[col], errors='coerce').fillna(0)

gdf = parcels.merge(account_values, on='ACCOUNT_NO', how='left')
for col in ['LANDACT', 'IMPACT', 'TOTALACT', 'LANDASD', 'IMPASD', 'TOTALASD']:
    gdf[col] = pd.to_numeric(gdf.get(col), errors='coerce').fillna(0)

print(f'Merged gdf: {len(gdf):,} parcels')
print(f'Total assessed: ${gdf["TOTALASD"].sum():,.0f}')

## Section 3 — Property classification

In [ ]:
def categorize(row):
    acct = str(row.get('ACCOUNT_TYPE_CODE') or '').upper().strip()
    desc = str(row.get('Valuation_Description') or '').upper()
    if acct == 'RESIDENTIAL':
        return 'Residential'
    if acct == 'COMMERCIAL':
        return 'Commercial'
    if acct == 'INDUSTRIAL':
        return 'Industrial'
    if acct == 'VACANT LAND' or 'VAC' in desc:
        return 'Vacant / Undeveloped'
    if acct in {'EXEMPT', 'UTILITIES', 'HOA', 'POSSESSORY INT'}:
        return 'Exempt / Government'
    if 'AGRIC' in desc:
        return 'Agricultural'
    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)
print(gdf['PROPERTY_CATEGORY'].value_counts())

# Exclude Exempt/Government and Agricultural from the revenue-neutral solve
gdf['excluded_from_solve'] = gdf['PROPERTY_CATEGORY'].isin(['Exempt / Government', 'Agricultural'])
gdf['full_exmp'] = gdf['excluded_from_solve'].astype(int)
print(f'\nExcluded from solve: {gdf["excluded_from_solve"].sum():,}')
print(f'Solve population:    {(~gdf["excluded_from_solve"]).sum():,}')

## Section 4 — Current tax (per-parcel mill levy)

Uses Douglas County's `REDUCED_MILL_LEVY` field per parcel (varies across HR's
~20 tax districts). Parcels missing the field fall back to 93.1 mills — the
2024 Douglas County certified rate for a typical HR residential district
(county + law enforcement + Re-1 schools + debt + library + South Metro Fire +
HR Metro District + drainage).


In [ ]:
gdf['mill_levy'] = pd.to_numeric(gdf.get('REDUCED_MILL_LEVY'), errors='coerce')
null_count = gdf['mill_levy'].isna().sum()
gdf['mill_levy'] = gdf['mill_levy'].fillna(HR_DEFAULT_MILL_LEVY)
print(f'Mill levy — median: {gdf["mill_levy"].median():.3f}  '
      f'min: {gdf["mill_levy"].min():.3f}  '
      f'max: {gdf["mill_levy"].max():.3f}  '
      f'({null_count:,} parcels used default {HR_DEFAULT_MILL_LEVY})')

gdf['taxable_land_value'] = gdf['LANDASD'].clip(lower=0)
gdf['taxable_improvement_value'] = gdf['IMPASD'].clip(lower=0)
gdf['current_tax'] = (gdf['taxable_land_value'] + gdf['taxable_improvement_value']) * gdf['mill_levy'] / 1000.0
gdf.loc[gdf['full_exmp'] == 1, 'current_tax'] = 0.0

current_revenue = float(gdf['current_tax'].sum())
print(f'Current revenue (all parcels, excluding exempt): ${current_revenue:,.0f}')

## Section 5 — Split-rate model (2:1)

In [ ]:
solve_df = gdf[gdf['full_exmp'] == 0].copy()
print(f'Solving on {len(solve_df):,} parcels (excluded: {len(gdf) - len(solve_df):,})')

land_millage, improvement_millage, new_revenue, solve_df = model_split_rate_tax(
    df=solve_df,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=float(solve_df['current_tax'].sum()),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

excluded = gdf[gdf['full_exmp'] == 1].copy()
excluded['new_tax'] = 0.0
excluded['tax_change'] = 0.0
excluded['tax_change_pct'] = 0.0
gdf = pd.concat([solve_df, excluded]).sort_index()

print(f'\nLand millage:        {land_millage:.4f}')
print(f'Improvement millage: {improvement_millage:.4f}')
print(f'Revenue: ${new_revenue:,.0f} (target ${current_revenue:,.0f})')

In [ ]:
category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Highlands Ranch — 2:1 Split-Rate Tax Impact')

## Section 6 — Exploration chart

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, ax = plt.subplots(figsize=(10, 5))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
summary.plot.barh(ax=ax, color=np.where(summary < 0, '#228B22', '#8B0000'))
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Highlands Ranch — Median Tax Change % by Category (2:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f'Saved preview chart')

## Section 7 — Census join + export

In [ ]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, census_categories=['Residential'])
print('Done.')